# Image Transformation Pipeline with PyTorch
# Part 2: Essential Transformations

This notebook covers essential image transformations in PyTorch.

Topics covered:
- Geometric transformations (Resize, Crop, Flip, Rotation)
- Color and intensity transformations (ColorJitter, Grayscale)
- Advanced transformations (RandomErasing, GaussianBlur, Perspective)


In [ ]:
# =========================================================================
# SETUP AND IMPORTS
# =========================================================================

import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.transforms import functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import requests
from io import BytesIO

print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")

## 1. Geometric Transformations

Geometric transformations modify the spatial properties of images:
- Resize: Change image dimensions
- Crop: Extract portions of the image
- Flip: Mirror the image
- Rotation: Rotate by an angle
- Affine: Apply affine transformations (rotation, translation, scale, shear)


In [ ]:
# Load sample image and helper to show multiple images

def load_sample_image():
    """Load a sample image, preferring a local copy in Week07/img."""
    local_path = Path("img/Cat03.jpg")
    if local_path.exists():
        return Image.open(local_path)

    # Fallback to downloading if the local copy is missing
    url = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/481px-Cat03.jpg"
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return Image.open(BytesIO(response.content))

original_img = load_sample_image()
print(f"Original image size: {original_img.size}")


def show_images(images, titles, rows=1, figsize=(15, 5)):
    """Display multiple images in a grid."""
    cols = (len(images) + rows - 1) // rows  # ceil division
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).reshape(rows, cols)

    for idx, (img, title) in enumerate(zip(images, titles)):
        row = idx // cols
        col = idx % cols

        # Convert tensor to PIL if needed
        if isinstance(img, torch.Tensor):
            img = img.permute(1, 2, 0).numpy()
            img = np.clip(img, 0, 1)

        axes[row, col].imshow(img)
        axes[row, col].set_title(title)
        axes[row, col].axis('off')

    plt.tight_layout()
    plt.show()


### 1.1 Resize Transformations

Resize changes image dimensions while optionally preserving aspect ratio.

Parameters:
- `size`: int or tuple `(height, width)`
- `interpolation`: `PIL.Image.NEAREST`, `BILINEAR`, `BICUBIC`, `LANCZOS`


In [ ]:
# Different resize modes

resize_224 = transforms.Resize(224)                  # Resize shorter edge to 224
resize_square = transforms.Resize((224, 224))        # Resize to exact square
resize_aspect = transforms.Resize(224, max_size=300) # Limit max dimension

images = [
    original_img,
    resize_224(original_img),
    resize_square(original_img),
    resize_aspect(original_img),
]

titles = [
    f"Original {original_img.size}",
    f"Resize(224) {resize_224(original_img).size}",
    f"Resize((224,224)) {resize_square(original_img).size}",
    f"Resize(224, max_size=300) {resize_aspect(original_img).size}",
]

show_images(images, titles, figsize=(16, 4))


### 1.2 Crop Transformations

Crop transformations extract portions of the image:
- `CenterCrop`: Extract center region (deterministic)
- `RandomCrop`: Extract random region (stochastic)
- `FiveCrop`: Extract 4 corners + center
- `TenCrop`: FiveCrop + flipped versions
- `RandomResizedCrop`: Random crop then resize (common in training)


In [ ]:
# Center Crop

center_crop = transforms.CenterCrop((200, 200))

# Random Crop (will show 3 different random crops)
random_crop = transforms.RandomCrop((200, 200))

# RandomResizedCrop (crops random portion and resizes)
random_resized = transforms.RandomResizedCrop(
    size=224,
    scale=(0.5, 1.0),  # Crop 50-100% of original
    ratio=(0.75, 1.33)  # Aspect ratio range
)

images = [
    original_img,
    center_crop(original_img),
    random_crop(original_img),
    random_crop(original_img),
    random_resized(original_img),
]

titles = [
    f"Original {original_img.size}",
    f"CenterCrop(200) {center_crop(original_img).size}",
    f"RandomCrop 1 {random_crop(original_img).size}",
    f"RandomCrop 2 {random_crop(original_img).size}",
    f"RandomResizedCrop {random_resized(original_img).size}",
]

show_images(images, titles, figsize=(16, 4))

# FiveCrop example
five_crop = transforms.FiveCrop(150)
five_crops = five_crop(original_img)

print(f"FiveCrop produces {len(five_crops)} crops")
show_images(
    five_crops,
    ["Top-Left", "Top-Right", "Bottom-Left", "Bottom-Right", "Center"],
    figsize=(16, 4),
)


### 1.3 Flip Transformations

Flip transformations mirror the image:
- `RandomHorizontalFlip`: Flip horizontally with probability `p`
- `RandomVerticalFlip`: Flip vertically with probability `p`
- Both are commonly used in data augmentation.


In [ ]:
# Flip transforms

h_flip = transforms.RandomHorizontalFlip(p=1.0)
v_flip = transforms.RandomVerticalFlip(p=1.0)
random_h_flip = transforms.RandomHorizontalFlip(p=0.5)

images = [
    original_img,
    h_flip(original_img),
    v_flip(original_img),
    random_h_flip(original_img),
    random_h_flip(original_img),
    random_h_flip(original_img),
]

titles = [
    "Original",
    "Horizontal Flip",
    "Vertical Flip",
    "Random H-Flip 1",
    "Random H-Flip 2",
    "Random H-Flip 3",
]

show_images(images, titles, rows=2, figsize=(16, 8))


### 1.4 Rotation and Affine Transformations

Rotation and affine transformations modify spatial geometry:
- `RandomRotation`: Rotate by random angle within range
- `RandomAffine`: Apply random affine transformation


In [ ]:
# Random rotation and affine

rotation = transforms.RandomRotation(degrees=30)
affine = transforms.RandomAffine(
    degrees=30,            # Rotation range
    translate=(0.1, 0.1),  # Translation range (% of image size)
    scale=(0.8, 1.2),      # Scale range
    shear=10,              # Shear angle
)

images = [
    original_img,
    rotation(original_img),
    rotation(original_img),
    affine(original_img),
    affine(original_img),
    affine(original_img),
]

titles = [
    "Original",
    "Random Rotation 1",
    "Random Rotation 2",
    "Random Affine 1",
    "Random Affine 2",
    "Random Affine 3",
]

show_images(images, titles, rows=2, figsize=(16, 8))


## 2. Color and Intensity Transformations

Color transformations modify pixel intensities and color properties:
- ColorJitter: Random brightness, contrast, saturation, hue
- Grayscale: Convert to grayscale
- RandomGrayscale: Random conversion to grayscale
- RandomInvert: Invert colors randomly
- RandomPosterize: Reduce number of bits per channel
- RandomSolarize: Invert pixels above threshold
- RandomEqualize: Equalize image histogram


### 2.1 Color Jitter

`ColorJitter` randomly changes brightness, contrast, saturation, and hue.

Parameters:
- `brightness`: How much to jitter brightness (0 = no change, 1 = max change)
- `contrast`: How much to jitter contrast
- `saturation`: How much to jitter saturation
- `hue`: How much to jitter hue (0.5 = max, meaning ±180°)


In [ ]:
# Different levels of ColorJitter

jitter_mild = transforms.ColorJitter(
    brightness=0.2,
    contrast=0.2,
    saturation=0.2,
    hue=0.1,
)

jitter_strong = transforms.ColorJitter(
    brightness=0.5,
    contrast=0.5,
    saturation=0.5,
    hue=0.3,
)

jitter_brightness = transforms.ColorJitter(brightness=0.5)
jitter_contrast = transforms.ColorJitter(contrast=0.5)

images = [
    original_img,
    jitter_brightness(original_img),
    jitter_brightness(original_img),
    jitter_contrast(original_img),
    jitter_contrast(original_img),
    jitter_mild(original_img),
    jitter_mild(original_img),
    jitter_strong(original_img),
    jitter_strong(original_img),
]

titles = [
    "Original",
    "Brightness 1",
    "Brightness 2",
    "Contrast 1",
    "Contrast 2",
    "Mild Jitter 1",
    "Mild Jitter 2",
    "Strong Jitter 1",
    "Strong Jitter 2",
]

show_images(images, titles, rows=3, figsize=(16, 12))


### 2.2 Grayscale Transformations

Grayscale transformations convert RGB images to grayscale:
- `Grayscale`: Always convert (deterministic)
- `RandomGrayscale`: Convert with probability `p`


In [ ]:
# Always grayscale

grayscale = transforms.Grayscale(num_output_channels=3)  # Keep 3 channels

# Random grayscale (p=1.0 for demonstration)
random_gray_always = transforms.RandomGrayscale(p=1.0)

# Random grayscale (p=0.5)
random_gray = transforms.RandomGrayscale(p=0.5)

images = [
    original_img,
    grayscale(original_img),
    random_gray(original_img),
    random_gray(original_img),
    random_gray(original_img),
]

titles = [
    "Original",
    "Grayscale (always)",
    "RandomGrayscale 1",
    "RandomGrayscale 2",
    "RandomGrayscale 3",
]

show_images(images, titles, figsize=(16, 4))


### 2.3 Advanced Color Transformations

Advanced transformations modify pixel values in interesting ways:
- `RandomInvert`: Invert colors with probability `p`
- `RandomPosterize`: Reduce bits per channel (creates poster effect)
- `RandomSolarize`: Invert pixels above threshold
- `RandomEqualize`: Equalize histogram


In [ ]:
# Advanced color transforms

invert = transforms.RandomInvert(p=1.0)
posterize = transforms.RandomPosterize(bits=4, p=1.0)
solarize = transforms.RandomSolarize(threshold=128, p=1.0)
equalize = transforms.RandomEqualize(p=1.0)

images = [
    original_img,
    invert(original_img),
    posterize(original_img),
    solarize(original_img),
    equalize(original_img),
]

titles = [
    "Original",
    "Inverted",
    "Posterized (4 bits)",
    "Solarized (t=128)",
    "Equalized",
]

show_images(images, titles, figsize=(16, 4))


## 3. Advanced Transformations

Advanced transformations for robust data augmentation:
- RandomErasing: Erase random rectangular region
- GaussianBlur: Apply Gaussian blur
- RandomPerspective: Apply random perspective transformation
- RandomAdjustSharpness: Adjust image sharpness
- RandomAutocontrast: Apply autocontrast


### 3.1 Random Erasing

`RandomErasing` randomly erases a rectangular region (cutout augmentation).

Parameters:
- `p`: Probability of applying
- `scale`: Range of proportion of erased area
- `ratio`: Range of aspect ratio of erased area
- `value`: Pixel value to fill (default is random)

Note: Must be applied to tensors, not PIL images.


In [ ]:
# Random erasing operates on tensors

to_tensor = transforms.ToTensor()
random_erasing = transforms.RandomErasing(
    p=1.0,              # Always erase for demonstration
    scale=(0.02, 0.15), # Erase 2-15% of image
    ratio=(0.3, 3.3),   # Aspect ratio range
    value="random",    # Fill with random values
)

# Apply transforms

tensor_img = to_tensor(original_img)

images = [
    tensor_img,
    random_erasing(tensor_img.clone()),
    random_erasing(tensor_img.clone()),
    random_erasing(tensor_img.clone()),
    random_erasing(tensor_img.clone()),
]

titles = [
    "Original",
    "Random Erasing 1",
    "Random Erasing 2",
    "Random Erasing 3",
    "Random Erasing 4",
]

show_images(images, titles, figsize=(16, 4))


### 3.2 Gaussian Blur

`GaussianBlur` applies Gaussian blur with random kernel size.

Parameters:
- `kernel_size`: Size of Gaussian kernel (odd number or range)
- `sigma`: Standard deviation (controls blur strength)


In [ ]:
# Different blur strengths

blur_mild = transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 1.0))
blur_medium = transforms.GaussianBlur(kernel_size=9, sigma=(0.1, 2.0))
blur_strong = transforms.GaussianBlur(kernel_size=15, sigma=(0.1, 3.0))

images = [
    original_img,
    blur_mild(original_img),
    blur_medium(original_img),
    blur_strong(original_img),
]

titles = [
    "Original",
    "Mild Blur (k=5)",
    "Medium Blur (k=9)",
    "Strong Blur (k=15)",
]

show_images(images, titles, figsize=(16, 4))


### 3.3 Random Perspective

`RandomPerspective` applies a random perspective transformation.

Parameters:
- `distortion_scale`: Controls the degree of distortion (0–1)
- `p`: Probability of applying the transformation


In [ ]:
# Different distortion levels
perspective_mild = transforms.RandomPerspective(distortion_scale=0.2, p=1.0)
perspective_medium = transforms.RandomPerspective(distortion_scale=0.4, p=1.0)
perspective_strong = transforms.RandomPerspective(distortion_scale=0.6, p=1.0)

images = [
    original_img,
    perspective_mild(original_img),
    perspective_medium(original_img),
    perspective_strong(original_img),
    perspective_strong(original_img),
    perspective_strong(original_img),
]

titles = [
    "Original",
    "Mild (0.2)",
    "Medium (0.4)",
    "Strong 1 (0.6)",
    "Strong 2 (0.6)",
    "Strong 3 (0.6)",
]

show_images(images, titles, rows=2, figsize=(16, 8))


### 3.4 Other Advanced Transformations

Additional useful transformations:
- `RandomAdjustSharpness`: Adjust image sharpness
- `RandomAutocontrast`: Apply autocontrast


In [ ]:
# Sharpness adjustment and autocontrast
sharpness_low = transforms.RandomAdjustSharpness(sharpness_factor=0.2, p=1.0)
sharpness_high = transforms.RandomAdjustSharpness(sharpness_factor=3.0, p=1.0)
autocontrast = transforms.RandomAutocontrast(p=1.0)

images = [
    original_img,
    sharpness_low(original_img),
    sharpness_high(original_img),
    autocontrast(original_img),
]

titles = [
    "Original",
    "Low Sharpness (0.2)",
    "High Sharpness (3.0)",
    "Autocontrast",
]

show_images(images, titles, figsize=(16, 4))


## Comprehensive Example: Combining Transformations

A realistic training pipeline often chains multiple transformations to increase data diversity.


In [ ]:

# Heavy augmentation pipeline
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]
to_tensor = transforms.ToTensor()
normalize = transforms.Normalize(mean=imagenet_mean, std=imagenet_std)

def denormalize(tensor_img):
    """Reverse normalization for display."""
    img = tensor_img.clone()
    for c in range(3):
        img[c] = img[c] * imagenet_std[c] + imagenet_mean[c]
    return torch.clamp(img, 0, 1)

heavy_augmentation = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
    to_tensor,
    normalize,
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15)),
])

print("Heavy Augmentation Pipeline:")
print("1. Resize to 256")
print("2. RandomResizedCrop to 224")
print("3. RandomHorizontalFlip (p=0.5)")
print("4. RandomRotation (±15°)")
print("5. ColorJitter")
print("6. RandomGrayscale (p=0.1)")
print("7. GaussianBlur")
print("8. ToTensor")
print("9. Normalize (ImageNet)")
print("10. RandomErasing (p=0.3)")

num_augmentations = 12
augmented_images = [heavy_augmentation(original_img) for _ in range(num_augmentations)]

# Add original for comparison (normalized to match augmented samples)
original_tensor = normalize(to_tensor(transforms.Resize(224)(original_img)))
all_images = [original_tensor] + augmented_images
all_titles = ["Original"] + [f"Augmented {i+1}" for i in range(num_augmentations)]

# Denormalize for display
all_images_display = [denormalize(img) for img in all_images]

show_images(all_images_display, all_titles, rows=3, figsize=(18, 12))


## Practical Guidelines: When to Use Which Transformation

GEOMETRIC TRANSFORMATIONS:
- ✓ RandomResizedCrop: Almost always use in training (very effective)
- ✓ RandomHorizontalFlip: Use when horizontal orientation doesn't matter
- ✗ RandomVerticalFlip: Rarely use (except specific cases like satellite imagery)
- ✓ RandomRotation: Use when rotation doesn't change meaning (not for text/digits)
- ✓ RandomAffine: More general than rotation, but computationally expensive

COLOR TRANSFORMATIONS:
- ✓ ColorJitter: Very effective, use in most cases
- ✓ RandomGrayscale: Use sparingly (`p=0.1` is common)
- ✗ Normalize: ALWAYS use (but not for augmentation, for preprocessing)
- ? Posterize/Solarize: Experimental, can hurt performance if overused

ADVANCED TRANSFORMATIONS:
- ✓ GaussianBlur: Effective for robustness
- ✓ RandomErasing: Very effective (similar to cutout)
- ? RandomPerspective: Can be too aggressive, use with small distortion
- ✓ AutoAugment/RandAugment: State-of-the-art, but complex

TYPICAL TRAINING PIPELINE:
1. Resize (larger than target)
2. RandomResizedCrop (to target size)
3. RandomHorizontalFlip
4. ColorJitter (mild to moderate)
5. Optional: Blur, Grayscale (low probability)
6. ToTensor
7. Normalize
8. RandomErasing

VALIDATION PIPELINE (No augmentation!):
1. Resize
2. CenterCrop
3. ToTensor
4. Normalize


## Key Takeaways - Part 2

1. Geometric transformations are essential:
   - RandomResizedCrop combines crop + resize (very effective)
   - RandomHorizontalFlip is almost universal
   - Rotation/Affine when orientation invariance is needed

2. Color transformations add diversity:
   - ColorJitter is highly effective
   - Use moderate values to avoid unrealistic images
   - Grayscale/Invert can help but use sparingly

3. Advanced transformations for robustness:
   - RandomErasing (cutout) prevents overfitting on local patterns
   - GaussianBlur adds robustness to image quality
   - Perspective can be too aggressive

4. Compose multiple transformations:
   - Order matters (geometric → color → tensor → normalize)
   - Start conservative; increase augmentation if overfitting
   - Different augmentation strength for different dataset sizes

5. Training vs validation:
   - Training: Use random/stochastic transforms
   - Validation: Use only deterministic transforms
   - Never augment validation/test data!
